# Exploração Histórica da Otimização Federada

Este notebook analisa o histórico de simulação salvo pelo `FLPOPT`. Os dados já estão formatados em DataFrames separados (`df_inputs` e `df_chosen`) para tornar a exploração mais intuitiva e isolada.

In [ ]:
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

with open('historico_20_rodadas.json', 'r') as f:
    history = json.load(f)

print(f"Número de rodadas na simulação: {len(history['chosen_solutions'])}")

## 1. Carregamento dos DataFrames Principais
Extraindo os `inputs` globais e a solução escolhida (`chosen_solutions`) de cada rodada.

In [ ]:
df_inputs = pd.DataFrame(history['inputs'])

chosen_records = []
for r, sol in enumerate(history['chosen_solutions']):
    record = {'Rodada': r, 'idx_fronteira': sol.get('idx', np.nan)}
    
    # Objetivos
    if sol['F'] is not None:
        record['f1_energia'] = sol['F'][0]
        record['f2_unselected'] = sol['F'][1]
        record['f3_tempo'] = sol['F'][2]
        
    # Variáveis de decisão X
    for k, v in sol['X'].items():
        record[k] = v
        
    chosen_records.append(record)

df_chosen = pd.DataFrame(chosen_records)

N_devices = len(df_inputs['c'][0])
c_vals = np.array(df_inputs['c'][0])
S_vals = np.array(df_inputs['S'][0])

df_chosen.head()

## 2. Histórico de Punição: `unselected_count`
Analisando quantas rodadas cada dispositivo passou sem ser escolhido para o treinamento.

In [ ]:
unselected_list = [inp['unselected_count'] for inp in history['inputs']]
df_unselected = pd.DataFrame(unselected_list, columns=[f'Device {n}' for n in range(N_devices)])

plt.figure(figsize=(12, 6))
sns.heatmap(df_unselected.T, cmap='YlOrRd', annot=True, cbar_kws={'label': 'Rodadas não selecionado'})
plt.title('Evolução do Unselected Count por Dispositivo e Rodada')
plt.xlabel('Rodada')
plt.ylabel('Dispositivo')
plt.show()

## 3. Evolução das Variáveis de Decisão ($f_n$ e $\psi_n$)
Visualizando o comportamento da frequência e número de épocas apenas para os dispositivos que foram **selecionados** na respectiva rodada.

In [ ]:
df_f = pd.DataFrame({'Rodada': df_chosen['Rodada']})
df_psi = pd.DataFrame({'Rodada': df_chosen['Rodada']})

for n in range(N_devices):
    mask = df_chosen[f'beta_{n}'] == 1.0
    df_f[f'Device {n}'] = np.where(mask, df_chosen[f'f_{n}'], np.nan)
    df_psi[f'Device {n}'] = np.where(mask, df_chosen[f'psi_{n}'], np.nan)

ncols = 3
nrows = math.ceil(N_devices / ncols)

# Frequências
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(15, 4 * nrows))
axes_flat = axes.flatten() if N_devices > 1 else [axes]

for n in range(N_devices):
    ax = axes_flat[n]
    ax.plot(df_f['Rodada'], df_f[f'Device {n}'], marker='o', label='Frequência', color='blue')
    ax.set_title(f'Evolução da Frequência - Dispositivo {n}')
    ax.set_xlabel('Rodada')
    ax.set_xticks(df_f['Rodada'].unique())
    ax.grid(True, linestyle=':', alpha=0.6)

for n in range(N_devices, len(axes_flat)):
    fig.delaxes(axes_flat[n])
plt.tight_layout()
plt.show()

# Épocas
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(15, 4 * nrows))
axes_flat = axes.flatten() if N_devices > 1 else [axes]

for n in range(N_devices):
    ax = axes_flat[n]
    ax.plot(df_psi['Rodada'], df_psi[f'Device {n}'], marker='X', label='Épocas (psi)', color='orange')
    ax.set_title(f'Evolução de Épocas (psi) - Dispositivo {n}')
    ax.set_xlabel('Rodada')
    ax.set_xticks(df_psi['Rodada'].unique())
    ax.grid(True, linestyle=':', alpha=0.6)

for n in range(N_devices, len(axes_flat)):
    fig.delaxes(axes_flat[n])
plt.tight_layout()
plt.show()

## 4. Análise de Limites: Tempo da Rodada ($T$) vs Tempo de Treinamento Local
Verifica se o tempo de treinamento dos dispositivos está atingindo a variável restritiva $T$ definida pelo otimizador.

In [ ]:
times_list = []
psi_comp_list = []
theta_means = []

for r in range(len(df_chosen)):
    row = df_chosen.iloc[r]
    T_val = row['T']
    
    theta_r = []
    for n in range(N_devices):
        if row[f'beta_{n}'] == 1.0:
            f_n = row[f'f_{n}']
            psi_n = row[f'psi_{n}']
            theta_n = row[f'theta_{n}']
            
            # Tempo local do dispositivo
            f_n_hz = f_n * 1e9
            t_n = (psi_n * c_vals[n] * S_vals[n]) / f_n_hz
            times_list.append({'Rodada': r, 'Dispositivo': n, 'Tempo': t_n})
            
            # Comparação de Psi mínimo
            psi_min = -np.log2(1 - theta_n)
            psi_comp_list.append({'Rodada': r, 'Dispositivo': n, 'psi': psi_n, 'Psi_min': psi_min})
            
            theta_r.append(theta_n)
            
    theta_means.append({'Rodada': r, 'Theta_Medio': np.mean(theta_r) if theta_r else np.nan})

df_times = pd.DataFrame(times_list)
df_psi_comp = pd.DataFrame(psi_comp_list)
df_theta = pd.DataFrame(theta_means)

plt.figure(figsize=(12, 6))
sns.scatterplot(data=df_times, x='Rodada', y='Tempo', hue='Dispositivo', palette='tab10', s=100, marker='o', alpha=0.7)
plt.plot(df_chosen['Rodada'], df_chosen['T'], color='red', linewidth=2, label='Tempo da Rodada (T)', linestyle='--')
plt.title('Tempo de Treinamento dos Dispositivos Selecionados vs Tempo da Rodada (T)')
plt.ylabel('Tempo (s)')
plt.xlabel('Rodada')
plt.xticks(df_chosen['Rodada'].unique())
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

## 5. Análise de Limites: Épocas exigidas ($\Psi_{min}$) e Fator de Acurácia ($\theta$)

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df_psi_comp, x='Rodada', y='psi', hue='Dispositivo', palette='tab10', s=100, marker='X', alpha=0.8, legend=False)
sns.scatterplot(data=df_psi_comp, x='Rodada', y='Psi_min', hue='Dispositivo', palette='tab10', s=50, marker='o', alpha=0.5)
plt.title('Comparação: psi_n escolhido (X) vs Psi_min exigido (o)')
plt.ylabel('Número de Épocas')
plt.xlabel('Rodada')
plt.xticks(df_chosen['Rodada'].unique())
plt.legend(title='Dispositivo', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(df_theta['Rodada'], df_theta['Theta_Medio'], marker='s', color='purple', linewidth=2)
plt.title('Evolução do Fator de Acurácia (theta_n) Médio (Dispositivos Selecionados)')
plt.ylabel('Theta Médio')
plt.xlabel('Rodada')
plt.xticks(df_theta['Rodada'].unique())
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()